# ☎️ Sprint 11 - Aprendizaje Supervisado

#### 🐤 Observaciones para él/la calificador/a

Buenos días/tardes/noches, calificador/a de mi proyecto 🐥

Para este proyecto utilicé librerías alternativas (Polars, Pydantic, Numpy) para la obtención, análisis de datos y modelado, por lo que el código no es ejecutable directamente en el entorno estándar del bootcamp (debido a polars). 

He incluido capturas de pantalla con los resultados obtenidos y las conclusiones correspondientes en caso de no poder verlas.

Si desea consultar los analísis generados (formato TXT y JSON), pueden encontrarse en la siguiente carpeta:  
- 📁 [Carpeta Reportes](../eda_analysis/analysis_2026-07-11/)
- 🧾 [TXT report](../eda_analysis/analysis_2026-07-11/04-33-26/TXT_report.txt)
- 📑 [JSON report](../eda_analysis/analysis_2026-07-11/04-33-26/JSON_analysis.json)

Quedo atenta a cualquier comentario. ¡Gracias por su revisión! 🌱

# ❗️❗️⚠️ Esto no sirve con las restructuraciones ⚠️❗️❗️
- Esta en constante cambio, por lo que tener en cuenta que no correran con los modulos importados pues fueron cambiados

In [ ]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.validation.read_validation import ReadConfig
from src.get_frame import get_frame
from src.eda.pipeline_eda import EdaPipeline
from src.cleaning.pipeline import CleanDataFrame
from src.ml_process.preprocessing.pipeline import AutoPipeline

import json
import polars as pl

## 📊 Preparación de Datos

### ⚙️ Configuraciones

In [ ]:
config= ReadConfig().read_config()

config_var= config['config_vars']
config_preprocessing= config['preprocessing']
config_modeling= config['modeling']
config_cleaning= config['cleaning']
config= config['config']

![](../assets/sprint11/config.png)

### 📊 DataFrame y EDA

In [ ]:
frame= get_frame(file=config.path.data)

dict_eda_files= EdaPipeline(frame=frame, config=config, config_var=config_var).pipeline_eda()
json_path= dict_eda_files['JSON_path']

![](../assets/sprint11/Frame.png)

#### 📑 EDA
- Hay 14 columnas y 10 mil filas
- 11 son columnas tipo númericas (entero y flotante) y las 3 restantes son strings (tipo categóricas)
- No hay valores nulos

#### 📝 Analísis de Datos
- 📊 Distribución 
    - Para las columnas: RowNumber, CreditScore, Tenure, Balance, HasCrCard, IsActiveMember y EstimatedSalary tiene una cola negativa es decir pocos de muchos
    - Para las columnas: Exited, NumOfProducts, Age, CustomerId tiene una cola positiva es decir muchos de pocos
    - La concentración varia segun la columnas pero por observaciones se puede notar que hay más jovenes que adultos lo cual ya puede darnos una pista de porque tantas personas lo estan dejando

- 🚨 Outliers: 
    - Se encontraron muchos valores atipicos en la columna Exited con un ~20% de desbalanceo en alguna clase... de los cuales explicarian tambien la cola a la derecha y no a la izquierda y por lo tanto su concentración (esto es desbalanceo puro en la columna target, no es una hipótesis es literalmente un terrible desbalanceo, Dios mío, igual si es debalanceo no es outlier, la cuestión es que mi reporte lo marcho justamente como outlier por desbalanceo)
    - Las demás columnas: CreditScore, Age, NumOfProducts; tienen un porcentaje de outliers pequeño, realmente no afecta o no debería afectar tanto la distribución de nuestro análisis arriba 

- 🔗 Correlación
    - No se encontraron correlaciones entre columnas numéricas

- 🧮 Categóricas 
    - Surname tiene demasiados valores unicos (alta cardinalidad) pero no con un umbral considerado alto para agrupar esos valores raros
    - Geography y Gender tienen baja cardinalidad con un umbral donde se considera sea agrupar o usar un cieto tipo de encoder

Reporte completo: 
📁 [TXT Report](../eda_analysis/analysis_2026-07-11/04-33-26/TXT_report.txt)

### 🧽 Clean DataFrame

In [ ]:
with open(json_path, 'r', encoding='utf-8') as f: 
    file= json.load(f)

frame_cleaned= CleanDataFrame(
    frame=frame, 
    config=config, 
    config_clean=config_cleaning, 
    JSON=file
).clean_dataframe()

![](../assets/sprint11/CleaningFrame.png)

🧴 Se limpiaron los valores nulos

### 🧹 Feature Engineering

In [ ]:
with open(json_path, 'r', encoding='utf-8') as f: 
    file= json.load(f)

frame= frame.with_row_index()

preprocessing= AutoPipeline(frame=frame, analysis=file, config=config, config_pre=config_preprocessing)

pre_processing_frame= preprocessing.auto_frame_tests()

![](../assets/sprint11/FramePreProcessed.png)

#### 📊 Manejo de columnas para distribución 
Para manejar la distribución de los datos (puesto que afecta en su concentración y que datos más verá el modelo) se hizo un tipo de "algoritmo" (if/else) donde se analiza el sesgo de los datos y en base a eso se decide que operación se hará, sea donde sea positivo o negativo, si es positivo lo que hace es ver su media y su distancia de la cola y en base a relas de negocio (consultar le configuración para analísis) decidir que transformador sea mejor, ejemplo, la cola es muy grande y positiva -> log1 (para recortar la longitud de la cola) de otra forma un sqrt estaría bien (pues no necesitaríamos ser tan agresivos en el recorte de la cola)

#### 🚨 Manejo de columnas para outliers 
Para el manejo de outliers se tomo en cuenta en sí el porcentaje de outliers que había en cada columna, en base a eso se decidió que transformador usar, si usar un escalador, filtrado, imputación, flag o transformación (las reglas de negocio las puede consultar en el archivo de config de preprocesamiento)

#### 🔗 Manejo de columnas para distribución 
En el caso de las correlaciones altas para evitar multicolinealidad también se tomo en cuenta el contexto de negocio (consultar la configuración para análisis). 

#### Manejo de columnas categóricas
Las columnas categóricas en este caso no fueron tratadas debido a su baja sea rareza de total de clases y tambien por baja cardinalidad, la cardinalidad al ser casi pareja hace a que no necesitemos agrupar por lo que no se aplico en este caso para este dataset un preprocesamiento para mis columnas categóricas

Relas de negocio Distribución y Correlación YAML: 
- ⚙️ [Config EDA Analísis](../config/config_analysis_values.yml)

Reglas de negocio Outlier YAML: 
- ⚙️ [Config PreProcesamiento](../config/config_preprocessing.yml)

Reporte completo: 
- 📁 [JSON Analísis](../eda_analysis/analysis_2026-07-11/04-33-26/JSON_analysis.json)

## 🪆 Modelado

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from src.ml_process.modeling.scaler import SelectScaler 

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.dummy import DummyClassifier
from sklearn.utils import shuffle

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import numpy as np

In [ ]:
random_state= config.ml_training.random_state
mh= config_modeling.models_hyperparameters

target= config.ml_training.target
test_size= config.ml_training.train_test_search

cv= config_modeling.grid_search_cv.cv
n_jobs= config_modeling.grid_search_cv.n_jobs

In [ ]:
best_f1= 0.2
auc_roc= 0.75
overfitting= 10

#### 🔍 Busqueda de mejores hiperparametros y modelo

In [ ]:
frame= pre_processing_frame['sample']

In [ ]:
scaler= SelectScaler(frame=frame.select(pl.selectors.numeric()), config_ml=config_modeling).auto()

In [ ]:
total_rows= frame.height
total_class= frame.filter(pl.col(target)==1).height

percent_1= round(total_class/total_rows, 3)*100
print(f'Porcentaje total de clientes que no se han ido: {abs(percent_1-100)}%')
print(f'Porcentaje total de clientes que se han ido: {percent_1}%')

![](../assets/sprint11/PorcentajeCliente.png)

In [ ]:
model_hyperparameters= {
    'decision_tree': {
        'model': DecisionTreeClassifier(random_state=random_state, class_weight='balanced'), 
        'params': {
            'max_depth': mh.decision_tree.max_depth, 
            'min_samples_split': mh.decision_tree.min_samples_split
        }
    }
    ,
    'random_forest': {
        'model': RandomForestClassifier(random_state=random_state, class_weight='balanced_subsample'), 
        'params': {
            'n_estimators': mh.random_forest.n_estimators, 
            'max_depth': mh.random_forest.max_depth, 
            'min_samples_split': mh.random_forest.min_samples_split
        }
    },
    'logistic_regression' :{
        'model': LogisticRegression(random_state=random_state, class_weight='balanced'),
        'params': {
            'solver': ['lbfgs', 'liblinear'], 
        }
    }
}

scoring_gs_cv= [
    'accuracy',
    'precision_macro',
    'recall_macro',
    'f1_macro'
]

In [ ]:
x= frame.drop(target).to_pandas(use_pyarrow_extension_array=True)
y= frame[target].to_pandas(use_pyarrow_extension_array=True)

x, y= shuffle(x, y, random_state=random_state)

In [ ]:
x_train, x_search, y_train, y_search= train_test_split(
    x, y, test_size=test_size, random_state=random_state
)

In [ ]:
ohe_cols= ['Gender', 'Geography']
cat_preprocessor = ColumnTransformer([
    ('cat_ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ohe_cols)
], remainder='passthrough')

x_train= cat_preprocessor.fit_transform(x_train)
x_search= cat_preprocessor.transform(x_search)

if scaler: 
    x_train= scaler.fit_transform(x_train)
    x_search= scaler.transform(x_search)

In [ ]:
for model_name, dict_model in model_hyperparameters.items(): 
    print(f"--- {model_name} ---")
    
    model= dict_model['model']
    params= dict_model['params']
    
    baseline= DummyClassifier(strategy='uniform')
    baseline.fit(x_train, y_train)
    y_predbaseline= baseline.predict(x_search)
    baseline_acc= accuracy_score(y_search, y_predbaseline)
    
    for scoring in scoring_gs_cv:
        grid_search= GridSearchCV(
            model, 
            params, 
            cv=cv, 
            scoring=scoring,  
            n_jobs=n_jobs
        )
        
        grid_search.fit(x_train, y_train)
        proba_valid= grid_search.predict_proba(x_search)[:,1]
        
        y_pred= grid_search.predict(x_search)
        
        if scoring == 'accuracy': 
            print('\n-- Accuracy --')
            s1= grid_search.score(x_train, y_train) 
            s2= grid_search.score(x_search, y_search)
            acc= accuracy_score(y_search, y_pred)
            
            difference= s1 - s2
            
            if difference < overfitting: 
                print('📊 El modelo no tiene overfitting')
            else: 
                print('📈 El modelo tiene overfitting')
            print(f'📏 La diferencia entre train y test es de: {difference:.3f}')
            
            if acc > baseline_acc: 
                print('📊 El modelo no tiene underfitting')
            else: 
                print('📉 El modelo posee underfitting, es decir, es peor que el modelo "tonto" o de "piso"')
            print(f'📏 El modelo de piso tiene un accuracy de {baseline_acc:.3f}; mientras que el accuracy de predicción tiene un accuracy de {acc:.3f}')
        elif scoring == 'precision_macro':
            print('\n-- Precisión --')
            ps= precision_score(y_search, y_pred)
            print(f'📐 Precision: {ps:.3f}')
        elif scoring == 'recall_macro': 
            print('\n-- Sensibilidad --')
            rs= recall_score(y_search, y_pred)
            print(f'📍 Sensibilidad: {rs:.3f}')
        else: 
            best_f1_= 0
            best_threshold= 0
            
            print('\n-- F1-Score  --')
            for th in np.arange(0.01, 0.3, 0.05):
                predict_val= (proba_valid > th).astype(int)
                f1_s= f1_score(y_search, predict_val)
                
                if f1_s > best_f1: 
                    best_f1_= f1_s
                    best_threshold= th
            
            print(f' Threshold: {best_threshold}')
            print(f' Best F1-score: {best_f1_}\n')
            
            fs= f1_score(y_search, y_pred)
            print(f'⚖️ F1-Score: {fs:.3f}')
            
            print('\n-- Roc-Auc-Score --')
            ras= roc_auc_score(y_search, proba_valid)
            if ras < auc_roc: 
                print('⛓️‍💥 Las predicciones para retención de clientes esta teniendo muchos problemas para predecir si podemos retener un cliente')
            else: 
                print('🔗 Las predicciones para retención de clientes es precisa')
            print(f'📊 Roc-Auc: {ras:.3f}')
    
    be= grid_search.best_params_
    print(f'\n🪄 Best params: {be}')
    
    print()

![](../assets/sprint11/Hiper0.png)
![](../assets/sprint11/Hiper1.png)
![](../assets/sprint11/Hiper2.png)

#### 📐 Entrenamiento final

In [ ]:
frame= pre_processing_frame['frame']

In [ ]:
x= frame.drop(target).to_pandas(use_pyarrow_extension_array=True)
y= frame[target].to_pandas(use_pyarrow_extension_array=True)

x, y= shuffle(x, y, random_state=random_state)

In [ ]:
scaler= SelectScaler(frame=frame.select(pl.selectors.numeric()), config_ml=config_modeling).auto()

In [ ]:
x_train, x_test, y_train, y_test= train_test_split(
    x, y, random_state=random_state, test_size=test_size
) 

In [ ]:
ohe_cols= ['Gender', 'Geography']
cat_preprocessor = ColumnTransformer([
    ('cat_ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ohe_cols)
], remainder='passthrough')

x_train= cat_preprocessor.fit_transform(x_train)
x_test= cat_preprocessor.transform(x_test)

if scaler: 
    x_train= scaler.fit_transform(x_train)
    x_test= scaler.transform(x_test)

In [ ]:
baseline= DummyClassifier(strategy='uniform', random_state=random_state)

baseline.fit(x_train, y_train)
y_predbaseline= baseline.predict(x_test)

baseline_acc= accuracy_score(y_test, y_predbaseline)

In [ ]:
model= RandomForestClassifier(
    random_state=random_state, 
    class_weight='balanced', 
    max_depth=10, 
    min_samples_split=10, 
    n_estimators=100
)

model.fit(x_train, y_train)
y_pred= model.predict(x_test)
y_prob= model.predict_proba(x_test)

In [ ]:
accuracy= accuracy_score(y_test, y_pred)
s1= model.score(x_train, y_train)
s2= model.score(x_test, y_test)

if baseline_acc > accuracy: 
    print('📉 El modelo posee underfitting')
else: 
    print('✅ El modelo no tiene underfitting')

difference= s1 - s2
if difference > overfitting: 
    print('📈 El modelo posee overfitting')
else: 
    print('✅ El modelo no tiene overfitting')

![](../assets/sprint11/UnderfittingOverfitting.png)

In [ ]:
precision= precision_score(y_test, y_pred)
recall= recall_score(y_test, y_pred)

print(f'📐 Precisión: {precision:.2f}')
print(f'📏 Sensibilidad: {recall:.2f}')

![](../assets/sprint11/PrecisionSensibilidad.png)

In [ ]:
f1= f1_score(y_test, y_pred)

print(f'📍 F1: {f1:2f}')


![](../assets/sprint11/f1.png)

In [ ]:
roc_auc= roc_auc_score(y_test, y_pred)

print(f'📌 Roc-Auc: {roc_auc}')

![](../assets/sprint11/rocauc.png)

## 🐥 Conclusión 

- DataFrame y EDA 
    - Se encontro distribucion concentradas a más jovenes que a adultos 
    - Se detectaron pocos outliers 
    - No hubo correlaciones 
    - Y columnas categóricas con muchos valores unicos (alta cardinalidad) pero valores raros ninguno y algunas con baja cardinalidad

- Clean DataFrame 
    - Valores nulos fueron detectados y por lo tanto limpiados

- Feature Engineer 
    - Para algunas columnas numericas con pocos outliers se agregaron transformadores para disminuir el tamaño de cola como sqrt y square en caso de ser el skew positivo

- Modelado 
    - Hiperparametros y mejor modelo 
        - El balanceo se manejo con class_weight 
        - El estudio del modelo se hizo probando diferentes modelos con diferentes hiperparametros buscando siempre el mejor 
        - El mejor modelo fue RandomForest con un 0.60 en su f1, sin overfitting y underfitting y ademas con un buen roc-auc de 0.8 (aprox) lo cual nos suguiere que el modelo no esta pasando por muchos falsos positivos para poder llegar al 1
    
    - Modelo Final
        - Es peor que el de prueba sampleado 🐤 pero con un f1-score de 0.59